# Lab 5 — Multi-Agent Deep Reinforcement Learning

**TDDE78 — Deep Reinforcement Learning**  
Linköping University, Spring 2026

---

## Overview

This lab extends deep RL to settings with **multiple interacting agents**. You will implement two CTDE (Centralized Training, Decentralized Execution) algorithms:

| Algorithm | Type | Critic input | Policy type |
|-----------|------|--------------|-------------|
| **MAPPO** | On-policy | Joint observations V(o₁…oₙ) | Shared actor (parameter sharing) |
| **MADDPG** | Off-policy | Joint obs + joint actions Q_i(o,a) | Per-agent actors (independent) |

Both follow the CTDE principle:
- **Decentralized execution**: each agent acts from its *own local observation* only
- **Centralized training**: critics see *all agents' observations (and actions)* for better credit assignment

### What you implement 
- `CentralizedCritic` — joint V(o₁, o₂, …, oₙ) for MAPPO  →  `networks.py`
- `MAPPOAgent.select_actions()` and `MAPPOAgent.update()`  →  this notebook
- `MADDPGAgents.select_actions()` and `MADDPGAgents.update()`  →  this notebook

### What is provided (do NOT re-implement)
- `DiscreteActor` from Lab 2 (shared across agents, MAPPO)
- `MADDPGActor` / `MADDPGCritic` — MADDPG network architectures
- `compute_gae`, `MultiAgentReplayBuffer`, plotting utilities
- `train_mappo` and `train_maddpg` training loops

### Environment
**simple_spread_v3** (PettingZoo) — 3 cooperative agents must spread to cover 3 landmarks.

| Property | Value |
|----------|-------|
| Agents | 3 (cooperative) |
| Obs dim | 18 per agent |
| Actions | 5 discrete |
| Episode length | 25 steps |

### Reference
Lowe et al., *"Multi-Agent Actor-Critic for Mixed Cooperative-Competitive Environments"*, NeurIPS 2017.

In [ ]:
# Install PettingZoo if not already available
import subprocess, sys
try:
    import pettingzoo
except ImportError:
    print("Installing pettingzoo...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pettingzoo[mpe]", "-q"])

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import os

# --- Resolve notebook directory and experiments output path ---
_here   = globals().get('__vsc_ipynb_file__', os.path.abspath(''))
_nb_dir = os.path.dirname(_here) if os.path.isfile(_here) else _here
if _nb_dir not in sys.path:
    sys.path.insert(0, _nb_dir)

EXPERIMENTS_DIR = os.path.normpath(os.path.join(_nb_dir, '..', 'experiments'))
print(f'Experiments directory: {EXPERIMENTS_DIR}')

from networks import (
    DiscreteActor, CentralizedCritic,
    MADDPGActor, MADDPGCritic, layer_init,
)
from utils import (
    compute_gae, MultiAgentReplayBuffer,
    plot_mappo_results, plot_maddpg_results, plot_comparison,
    _save_plot, smooth,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Environment constants
N_AGENTS   = 3
OBS_DIM    = 18    # simple_spread_v3 with N=3
ACTION_DIM = 5

def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

set_seed(42)
print("Setup complete!")

---

## A.0 — Explore the Environment

**simple_spread_v3** (PettingZoo MPE) — 3 cooperative agents must each reach one of 3 landmarks.

| Property | Value |
|----------|-------|
| Agents | 3 (`agent_0`, `agent_1`, `agent_2`) |
| Observation | 18-dim per agent (own vel + pos, landmark positions, other agents' positions) |
| Actions | 5 discrete (no-op, left, right, up, down) |
| Reward | Shared: −Σ min‖agent − landmark‖ per step |
| Episode length | 25 steps |

The task is **cooperative**: agents are rewarded as a team for collectively covering all landmarks.


In [ ]:
from pettingzoo.mpe import simple_spread_v3

env = simple_spread_v3.parallel_env(N=N_AGENTS, continuous_actions=False, max_cycles=25)
obs_dict, _ = env.reset(seed=42)

print(f"Agents       : {env.agents}")
print(f"Obs space    : {env.observation_space('agent_0')}  → dim {OBS_DIM}")
print(f"Action space : {env.action_space('agent_0')}  → {ACTION_DIM} discrete actions")
print(f"  0=no-op  1=left  2=right  3=up  4=down")

# Random rollout to see baseline performance
ep_reward = 0.0
for _ in range(25):
    actions = {a: env.action_space(a).sample() for a in env.agents}
    _, rewards, _, _, _ = env.step(actions)
    ep_reward += sum(rewards.values())
print(f"\nRandom agent — episode reward: {ep_reward:.1f}  (good policy ≈ −30)")
env.close()

## Part A: Implementation

### A.1 — MAPPO
**Step 1:** Implement `CentralizedCritic` in `networks.py`  
**Step 2:** Implement `MAPPOAgent.select_actions()` and `MAPPOAgent.update()` below

### A.2 — MADDPG
**Step 3:** Implement `MADDPGAgents.select_actions()` and `MADDPGAgents.update()` below  
(Networks `MADDPGActor` and `MADDPGCritic` are provided — no changes to `networks.py`)

In [ ]:
class MAPPOAgent:
    """
    Multi-Agent PPO with parameter sharing and centralized critic (CTDE).

    Architecture:
    - Shared DiscreteActor: one set of weights used by ALL agents
    - CentralizedCritic: sees joint observations (all agents concatenated)

    Training:
    - Collect rollout_episodes full-episode rollouts (concatenated along time)
    - Normalize rewards globally across the whole batch before computing GAE
    - n_epochs PPO updates on shared actor and centralized critic
    """

    def __init__(
        self,
        obs_dim: int,
        action_dim: int,
        n_agents: int,
        critic=None,
        lr: float = 3e-4,
        gamma: float = 0.99,
        gae_lambda: float = 0.95,
        clip_eps: float = 0.2,
        vf_coef: float = 0.5,
        ent_coef: float = 0.001,
        n_epochs: int = 4,
        batch_size: int = 64,
        max_grad_norm: float = 0.5,
    ):
        self.obs_dim       = obs_dim
        self.action_dim    = action_dim
        self.n_agents      = n_agents
        self.gamma         = gamma
        self.gae_lambda    = gae_lambda
        self.clip_eps      = clip_eps
        self.vf_coef       = vf_coef
        self.ent_coef      = ent_coef
        self.n_epochs      = n_epochs
        self.batch_size    = batch_size
        self.max_grad_norm = max_grad_norm

        self.actor  = DiscreteActor(obs_dim, action_dim).to(device)
        self.critic = (critic if critic is not None
                       else CentralizedCritic(obs_dim, n_agents)).to(device)

        self.optimizer = torch.optim.Adam(
            list(self.actor.parameters()) + list(self.critic.parameters()), lr=lr
        )

    @torch.no_grad()
    def select_actions(self, obs_dict, agent_list):
        """
        Select actions for all agents (DECENTRALIZED execution).
        Each agent runs the SHARED actor on its OWN observation only.

        Also computes the centralized value estimate for this step
        (used for GAE during training).

        Args:
            obs_dict   (dict): {agent_id: np.array(obs_dim)}
            agent_list (list): ordered list of agent ids

        Returns:
            actions   (dict): {agent_id: int}
            log_probs (dict): {agent_id: float}
            value     (float): centralized V(joint_obs)

        Hints:
            1. For each agent, convert obs_dict[agent] to a FloatTensor (1, obs_dim)
               and call self.actor.get_action(obs_t) → (action, log_prob, entropy)
            2. Build joint_obs by np.concatenate-ing all agents' obs arrays,
               shape (n_agents * obs_dim,), then unsqueeze to (1, n_agents * obs_dim)
            3. Call self.critic(joint_obs_t) and extract the scalar value
        """
        # =====================================================================
        # TODO: Implement decentralized action selection + centralized value.
        # =====================================================================
        raise NotImplementedError("Implement MAPPOAgent.select_actions()")

    def update(self, obs_arr, actions_arr, rewards_arr, dones_arr,
               log_probs_arr, values_arr, last_value):
        """
        PPO update over a concatenated multi-episode rollout.

        Args:
            obs_arr      (np.ndarray): (T, n_agents, obs_dim)
            actions_arr  (np.ndarray): (T, n_agents)  — integer actions
            rewards_arr  (np.ndarray): (T, n_agents)  — per-agent rewards
            dones_arr    (np.ndarray): (T,)            — episode-end flags
            log_probs_arr(np.ndarray): (T, n_agents)  — log π(a|o) at collection
            values_arr   (np.ndarray): (T,)            — centralized V(s) at collection
            last_value   (float):      V(s_T) bootstrap

        Returns:
            (policy_loss, value_loss) averaged over update steps

        Hints:
        Step 1 — Global reward normalisation:
            In simple_spread all agents share the same reward, so per-agent std ≈ 0.
            Normalise globally: r_mean = rewards_arr.mean(), r_std = rewards_arr.std()
            Only divide if r_std > 1e-6 to avoid NaN.

        Step 2 — Compute per-agent GAE:
            For each agent i (0..N-1), call compute_gae(
                rewards_arr[:, i], values_arr, dones_arr,
                last_value, self.gamma, self.gae_lambda
            ) → (advantages_i, returns_i); store in all_advantages[:, i], all_returns[:, i]

        Step 3 — Normalize advantages:
            flat_adv = all_advantages.flatten()
            Only divide if flat_adv.std() > 1e-6

        Step 4 — Prepare tensors:
            Actor data:  obs_flat (T*N, obs_dim), actions_flat (T*N,),
                         old_lp_flat (T*N,), adv_flat (T*N,)
            Critic data: joint_obs (T, N*D), mean_returns (T,) = all_returns.mean(axis=1)

        Step 5 — n_epochs PPO mini-batch updates:
            Actor clip loss:
                ratio = exp(new_log_prob - old_log_prob)
                L_clip = -min(ratio * adv, clamp(ratio, 1-eps, 1+eps) * adv).mean()
                         - ent_coef * entropy.mean()
            Critic MSE loss:
                L_value = F.mse_loss(self.critic(joint_obs[cb]).squeeze(-1), mean_returns[cb])
            Total: L_clip + vf_coef * L_value
            Clip gradients with max_grad_norm before optimizer.step()
        """
        # =====================================================================
        # TODO: Implement the PPO update loop.
        # =====================================================================
        raise NotImplementedError("Implement MAPPOAgent.update()")

In [ ]:
def train_mappo(env, agent, n_episodes=5000, rollout_episodes=10, print_every=500):
    """
    Train MAPPOAgent on a PettingZoo parallel environment.

    Fully provided — do NOT modify.

    Collects `rollout_episodes` full episodes before each PPO update,
    concatenated along the time axis. This gives larger, less noisy batches
    — key for stable MAPPO learning (mirrors epymarl's batch_size=10 episodes).

    Episode boundaries are preserved via the done flags so GAE does not
    bleed across episodes.

    Args:
        env              : PettingZoo parallel environment
        agent            : MAPPOAgent
        n_episodes       : Total number of training episodes
        rollout_episodes : Episodes collected per PPO update
        print_every      : Logging interval (episodes)

    Returns:
        results (dict): 'episode_rewards', 'policy_losses', 'value_losses'
    """
    results = {'episode_rewards': [], 'policy_losses': [], 'value_losses': []}
    episode = 0

    while episode < n_episodes:
        # --- Collect rollout_episodes episodes ---
        batch_obs, batch_actions, batch_rewards = [], [], []
        batch_dones, batch_log_probs, batch_values = [], [], []

        for _ in range(min(rollout_episodes, n_episodes - episode)):
            obs_dict, _ = env.reset()
            agent_list  = list(obs_dict.keys())

            ep_obs, ep_actions, ep_rewards   = [], [], []
            ep_dones, ep_log_probs, ep_values = [], [], []
            ep_total_reward = 0.0

            while env.agents:
                actions, log_probs, value = agent.select_actions(obs_dict, agent_list)

                obs_arr = np.array([obs_dict[a] for a in agent_list])
                ep_obs.append(obs_arr)
                ep_actions.append([actions[a]    for a in agent_list])
                ep_log_probs.append([log_probs[a] for a in agent_list])
                ep_values.append(value)

                next_obs, rewards, terminations, truncations, _ = env.step(actions)
                dones = {a: terminations.get(a, False) or truncations.get(a, False)
                         for a in agent_list}

                ep_rewards.append([rewards.get(a, 0.0) for a in agent_list])
                ep_dones.append(float(any(dones.values())))
                ep_total_reward += sum(rewards.get(a, 0.0) for a in agent_list)
                obs_dict = next_obs

            results['episode_rewards'].append(ep_total_reward)
            episode += 1

            batch_obs.append(np.array(ep_obs,        dtype=np.float32))
            batch_actions.append(np.array(ep_actions,   dtype=np.int64))
            batch_rewards.append(np.array(ep_rewards,   dtype=np.float32))
            batch_dones.append(np.array(ep_dones,     dtype=np.float32))
            batch_log_probs.append(np.array(ep_log_probs, dtype=np.float32))
            batch_values.append(np.array(ep_values,    dtype=np.float32))

        # --- Concatenate episodes along time axis and update once ---
        p_loss, v_loss = agent.update(
            np.concatenate(batch_obs,       axis=0),
            np.concatenate(batch_actions,   axis=0),
            np.concatenate(batch_rewards,   axis=0),
            np.concatenate(batch_dones,     axis=0),
            np.concatenate(batch_log_probs, axis=0),
            np.concatenate(batch_values,    axis=0),
            last_value=0.0,
        )
        results['policy_losses'].append(p_loss)
        results['value_losses'].append(v_loss)

        if episode % print_every == 0 or episode >= n_episodes:
            mean_r = np.mean(results['episode_rewards'][-print_every:])
            print(f"Episode {episode:5d} | Mean Team Reward: {mean_r:7.2f}")

    return results

### A.1 — Train MAPPO on simple_spread

Train 3 cooperative agents to spread across 3 landmarks.  
Verify your implementation produces a learning curve (reward should increase(slighly sometime) over training).

In [ ]:
set_seed(42)

env = simple_spread_v3.parallel_env(N=N_AGENTS, continuous_actions=False, max_cycles=25)

mappo_agent = MAPPOAgent(
    obs_dim    = OBS_DIM,
    action_dim = ACTION_DIM,
    n_agents   = N_AGENTS,
    critic     = CentralizedCritic(OBS_DIM, N_AGENTS),
    lr         = 3e-4,
    gamma      = 0.99,
    gae_lambda = 0.95,
    clip_eps   = 0.2,
    vf_coef    = 0.5,
    ent_coef   = 0.001,
    n_epochs   = 4,
    batch_size = 64,
)

results_mappo = train_mappo(env, mappo_agent, n_episodes=5000, rollout_episodes=10,
                            print_every=500)
env.close()

plot_mappo_results(results_mappo, title="MAPPO — simple_spread",
                  window=100, experiments_dir=EXPERIMENTS_DIR)
print(f"Final avg reward (last 100 ep): {np.mean(results_mappo['episode_rewards'][-100:]):.1f}")

### A.2 — MADDPG: Multi-Agent DDPG

MADDPG (Lowe et al., NeurIPS 2017) extends single-agent DDPG to multi-agent settings using CTDE:

**Key differences from MAPPO:**

| | MAPPO | MADDPG |
|---|---|---|
| Policy update | On-policy (PPO clip) | Off-policy (DPG via Gumbel-Softmax) |
| Actor | **Shared** weights across agents | **Independent** per-agent weights |
| Critic | V(o₁…oₙ) — value function | Q_i(o₁…oₙ, a₁…aₙ) — Q-function per agent |
| Data | Episode rollouts | Replay buffer |
| Target nets | ✗ | ✓ Soft-updated (Polyak averaging) |

**Training loop for each gradient step:**
1. **Critic update**: minimize Bellman TD error using target networks  
   `y_i = r_i + γ · Q_i^target(s', π_target(s'))` → MSE loss
2. **Actor update**: maximize `Q_i` via Gumbel-Softmax (differentiable discrete actions)  
   `L_i = −Q_i(s, [a_1^gs, …, a_N^gs])`  where only agent `i`'s action uses gradients
3. **Soft target update**: `θ_target ← τ·θ + (1−τ)·θ_target`

### A.2 — Train MADDPG on simple_spread

Train 3 cooperative agents using off-policy MADDPG.  
Verify your implementation produces a learning curve (reward should improve over training).

In [ ]:
class MADDPGAgents:
    """
    MADDPG: Multi-Agent DDPG with Centralized Critics (CTDE).

    Each agent i maintains four networks:
      - actor_i:         π_i(a_i | o_i)            local obs  → action probs
      - target_actor_i:  lagged copy of actor_i
      - critic_i:        Q_i(o, a)                 joint obs + joint actions → Q
      - target_critic_i: lagged copy of critic_i

    Off-policy training via a shared MultiAgentReplayBuffer.
    Discrete actions are handled via Gumbel-Softmax during policy gradient.

    CTDE principle:
      - EXECUTION : each agent acts from its OWN observation only (decentralized)
      - TRAINING  : each agent's critic sees ALL observations and actions (centralized)
    """

    def __init__(
        self,
        obs_dim: int,
        action_dim: int,
        n_agents: int,
        lr_actor: float = 1e-3,
        lr_critic: float = 1e-3,
        gamma: float = 0.99,
        tau: float = 0.005,      # soft target update rate (Polyak averaging)
        batch_size: int = 1024,
        buffer_capacity: int = 1_000_000,
        warmup_steps: int = 1000,
    ):
        self.obs_dim      = obs_dim
        self.action_dim   = action_dim
        self.n_agents     = n_agents
        self.gamma        = gamma
        self.tau          = tau
        self.batch_size   = batch_size
        self.warmup_steps = warmup_steps
        self.total_steps  = 0

        # Per-agent networks (provided — do NOT modify)
        self.actors         = [MADDPGActor(obs_dim, action_dim).to(device) for _ in range(n_agents)]
        self.target_actors  = [MADDPGActor(obs_dim, action_dim).to(device) for _ in range(n_agents)]
        self.critics        = [MADDPGCritic(obs_dim, action_dim, n_agents).to(device) for _ in range(n_agents)]
        self.target_critics = [MADDPGCritic(obs_dim, action_dim, n_agents).to(device) for _ in range(n_agents)]

        # Initialise target networks as exact copies
        for i in range(n_agents):
            self.target_actors[i].load_state_dict(self.actors[i].state_dict())
            self.target_critics[i].load_state_dict(self.critics[i].state_dict())

        self.actor_optims  = [torch.optim.Adam(self.actors[i].parameters(),  lr=lr_actor)  for i in range(n_agents)]
        self.critic_optims = [torch.optim.Adam(self.critics[i].parameters(), lr=lr_critic) for i in range(n_agents)]

        self.buffer = MultiAgentReplayBuffer(buffer_capacity, obs_dim, action_dim, n_agents)

    @torch.no_grad()
    def select_actions(self, obs_dict: dict, agent_list: list, explore: bool = True):
        """
        Select actions for all agents — DECENTRALIZED execution.
        Each agent uses only its OWN observation and its OWN actor.

        Args:
            obs_dict   (dict): {agent_id: np.array(obs_dim)}
            agent_list (list): ordered agent ids
            explore    (bool): sample if True, argmax if False

        Returns:
            actions    (dict): {agent_id: int}
            actions_oh (dict): {agent_id: np.array(action_dim)}  one-hot

        Hints:
            1. For each agent i, convert obs_dict[agent] to a FloatTensor (1, obs_dim)
            2. Call self.actors[i].get_action(obs_t, explore=explore)
               → returns (action_int, action_onehot)
            3. Store int in actions[agent], .squeeze(0).cpu().numpy() in actions_oh[agent]
        """
        # =====================================================================
        # TODO: Implement decentralized action selection.
        # =====================================================================
        raise NotImplementedError("Implement MADDPGAgents.select_actions()")

    def update(self):
        """
        One gradient update step for all N agents.

        Returns (None, None) if the replay buffer has fewer than batch_size transitions.

        For each agent i:
          1. Critic update — minimize Bellman TD error:
             a. Sample a batch from self.buffer
             b. With torch.no_grad(), compute target actions from target_actors:
                next_acts_list[j] = self.target_actors[j].get_probs(next_obs_b[:, j, :])
                next_joint_acts   = torch.cat(next_acts_list, dim=-1)  # (B, N*action_dim)
             c. Compute bootstrap target for agent i:
                target_q = self.target_critics[i](joint_next_obs, next_joint_acts)
                y = r_i + γ * target_q * (1 - done)
             d. Critic loss: F.mse_loss(self.critics[i](joint_obs, joint_acts), y)
             e. critic_optims[i].zero_grad() → backward → clip grad 0.5 → step

          2. Actor update — maximize Q_i via Gumbel-Softmax:
             a. Build gs_acts_list: for j == i use self.actors[j].gumbel_softmax_action(obs_b[:,j,:])
                                    for j != i use acts_b[:, j, :] from the buffer
             b. Actor loss: -self.critics[i](joint_obs, gs_joint_acts).mean()
             c. actor_optims[i].zero_grad() → backward → clip grad 0.5 → step

          3. Soft-update both actors and critics:
             θ_target ← τ·θ + (1−τ)·θ_target

        Returns:
            (mean_actor_loss, mean_critic_loss) across N agents

        Useful tensor shapes (B = batch_size, N = n_agents):
            obs_b, next_obs_b  : (B, N, obs_dim)
            acts_b             : (B, N, action_dim)
            rew_b              : (B, N)
            dones_b            : (B,)

            joint_obs          : obs_b.view(B, -1)      → (B, N*obs_dim)
            joint_acts         : acts_b.view(B, -1)     → (B, N*action_dim)
            joint_next_obs     : next_obs_b.view(B, -1) → (B, N*obs_dim)
        """
        if len(self.buffer) < self.batch_size:
            return None, None

        # =====================================================================
        # TODO: Implement the MADDPG update step.
        # =====================================================================
        raise NotImplementedError("Implement MADDPGAgents.update()")

In [ ]:
def train_maddpg(env, agents, n_episodes=3000, updates_per_step=1, print_every=300):
    """
    Train MADDPGAgents on a PettingZoo parallel environment.

    Fully provided — do NOT modify.

    Off-policy step-by-step collection. Rewards are normalised using a
    running mean/std (Welford online algorithm) before being stored in the
    replay buffer — this stabilises critic targets and prevents Q-value
    divergence.

    Returns:
        results (dict): 'episode_rewards', 'actor_losses', 'critic_losses'
    """
    results = {'episode_rewards': [], 'actor_losses': [], 'critic_losses': []}

    # Running reward statistics for normalisation (Welford online algorithm)
    rew_mean  = np.zeros(agents.n_agents, dtype=np.float64)
    rew_var   = np.ones(agents.n_agents,  dtype=np.float64)
    rew_count = 1e-4

    for episode in range(n_episodes):
        obs_dict, _     = env.reset()
        agent_list      = list(obs_dict.keys())
        ep_total_reward = 0.0

        while env.agents:
            actions, actions_oh = agents.select_actions(obs_dict, agent_list, explore=True)
            agents.total_steps += 1

            next_obs, rewards, terminations, truncations, _ = env.step(actions)
            dones = {a: terminations.get(a, False) or truncations.get(a, False)
                     for a in agent_list}
            done = any(dones.values())

            obs_arr      = np.array([obs_dict[a]         for a in agent_list])
            actions_arr  = np.array([actions_oh[a]       for a in agent_list])
            rewards_arr  = np.array([rewards.get(a, 0.0) for a in agent_list])
            next_obs_arr = np.array([next_obs.get(a, obs_dict[a]) for a in agent_list])

            # Running normalisation of rewards before storing in buffer
            batch_mean = rewards_arr.mean(axis=0)
            rew_count += 1
            delta      = batch_mean - rew_mean
            rew_mean  += delta / rew_count
            rew_var    = rew_var + delta * (batch_mean - rew_mean)
            rew_std    = np.sqrt(rew_var / rew_count + 1e-8)
            rewards_norm = (rewards_arr - rew_mean) / rew_std

            agents.buffer.push(obs_arr, actions_arr, rewards_norm, next_obs_arr, done)
            ep_total_reward += rewards_arr.sum()   # log raw reward

            if agents.total_steps > agents.warmup_steps:
                for _ in range(updates_per_step):
                    a_loss, c_loss = agents.update()
                    if a_loss is not None:
                        results['actor_losses'].append(a_loss)
                        results['critic_losses'].append(c_loss)

            obs_dict = next_obs

        results['episode_rewards'].append(ep_total_reward)

        if (episode + 1) % print_every == 0:
            mean_r = np.mean(results['episode_rewards'][-print_every:])
            print(f"Episode {episode+1:4d} | Mean Team Reward: {mean_r:7.2f} "
                  f"| Buffer: {len(agents.buffer):,}")

    return results

In [ ]:
set_seed(42)

env_maddpg = simple_spread_v3.parallel_env(N=N_AGENTS, continuous_actions=False, max_cycles=25)

maddpg_agents = MADDPGAgents(
    obs_dim         = OBS_DIM,
    action_dim      = ACTION_DIM,
    n_agents        = N_AGENTS,
    lr_actor        = 1e-3,
    lr_critic       = 3e-4,   # lower to prevent Q-value divergence
    gamma           = 0.99,
    tau             = 0.005,
    batch_size      = 1024,
    buffer_capacity = 1_000_000,
    warmup_steps    = 1000,
)

results_maddpg = train_maddpg(env_maddpg, maddpg_agents, n_episodes=3000, print_every=300)
env_maddpg.close()

plot_maddpg_results(results_maddpg, title="MADDPG — simple_spread",
                   window=100, experiments_dir=EXPERIMENTS_DIR)
print(f"Final avg reward (last 100 ep): {np.mean(results_maddpg['episode_rewards'][-100:]):.1f}")

## Part B: Experiments

Compare MAPPO and MADDPG across multiple seeds to understand when each approach excels.

### B.1 — MAPPO vs MADDPG

Both algorithms follow CTDE but differ fundamentally in their training paradigm:

| Property | MAPPO | MADDPG |
|----------|-------|--------|
| Sample efficiency | Lower (on-policy, discards data) | Higher (off-policy replay) |
| Coordination mechanism | Parameter sharing (shared actor) | Centralized Q-function per agent |
| Stability | More stable (PPO clipping) | Can be less stable (bootstrapping) |
| Scalability | Better with many agents (shared weights) | Grows linearly with agent count |

Run 3 seeds each to measure mean ± std of team reward.

In [ ]:
N_SEEDS      = 3
N_MAPPO_EPS  = 5000
N_MADDPG_EPS = 3000

all_results = {'MAPPO': [], 'MADDPG': []}

for seed in range(N_SEEDS):
    set_seed(seed)

    # ---- MAPPO ----
    env = simple_spread_v3.parallel_env(N=N_AGENTS, continuous_actions=False, max_cycles=25)
    ag  = MAPPOAgent(
        obs_dim    = OBS_DIM,
        action_dim = ACTION_DIM,
        n_agents   = N_AGENTS,
        critic     = CentralizedCritic(OBS_DIM, N_AGENTS),
        lr         = 3e-4,
        gamma      = 0.99,
        gae_lambda = 0.95,
        clip_eps   = 0.2,
        vf_coef    = 0.5,
        ent_coef   = 0.001,
        n_epochs   = 4,
        batch_size = 64,
    )
    res_mappo = train_mappo(env, ag, n_episodes=N_MAPPO_EPS, rollout_episodes=10,
                            print_every=N_MAPPO_EPS)
    all_results['MAPPO'].append(res_mappo)
    env.close()
    print(f"[Seed {seed}] MAPPO:  final mean = {np.mean(res_mappo['episode_rewards'][-100:]):.2f}")

    # ---- MADDPG ----
    env = simple_spread_v3.parallel_env(N=N_AGENTS, continuous_actions=False, max_cycles=25)
    maddpg = MADDPGAgents(
        obs_dim         = OBS_DIM,
        action_dim      = ACTION_DIM,
        n_agents        = N_AGENTS,
        lr_actor        = 1e-3,
        lr_critic       = 3e-4,
        gamma           = 0.99,
        tau             = 0.005,
        batch_size      = 1024,
        buffer_capacity = 1_000_000,
        warmup_steps    = 1000,
    )
    res_maddpg = train_maddpg(env, maddpg, n_episodes=N_MADDPG_EPS,
                              print_every=N_MADDPG_EPS)
    all_results['MADDPG'].append(res_maddpg)
    env.close()
    print(f"[Seed {seed}] MADDPG: final mean = {np.mean(res_maddpg['episode_rewards'][-100:]):.2f}")

print('\nMean +/- Std (last 100 episodes):')
for name, runs in all_results.items():
    final = [np.mean(r['episode_rewards'][-100:]) for r in runs]
    print(f"  {name}: {np.mean(final):.1f} +/- {np.std(final):.1f}")

In [ ]:
plot_comparison(
    all_results,
    title="B.1 — MAPPO vs MADDPG on simple_spread",
    window=20,
    experiments_dir=EXPERIMENTS_DIR,
)

---

## B.2 — MAPPO: Effect of Entropy Coefficient

The entropy bonus `ent_coef` controls exploration in MAPPO.
Too high → policy stays too random and never converges.
Too low → premature convergence to a suboptimal deterministic policy.

Sweep `ent_coef` ∈ {0.0, 0.001, 0.01, 0.05} over 3 seeds.

In [ ]:
ent_values = [0.0, 0.001, 0.01, 0.05]
seeds      = [0, 1, 2]

ent_results = {}
for ent in ent_values:
    ent_results[f'ent = {ent}'] = []
    for seed in seeds:
        set_seed(seed)
        env = simple_spread_v3.parallel_env(N=N_AGENTS, continuous_actions=False, max_cycles=25)
        ag  = MAPPOAgent(
            obs_dim    = OBS_DIM,
            action_dim = ACTION_DIM,
            n_agents   = N_AGENTS,
            critic     = CentralizedCritic(OBS_DIM, N_AGENTS),
            lr         = 3e-4,
            ent_coef   = ent,
        )
        res = train_mappo(env, ag, n_episodes=3000, print_every=3000)
        ent_results[f'ent = {ent}'].append(res)
        env.close()

plot_comparison(
    ent_results,
    title="B.2 — MAPPO Entropy Coefficient Ablation",
    window=100,
    experiments_dir=EXPERIMENTS_DIR,
)

print('Mean +/- Std (last 100 episodes):')
for name, runs in ent_results.items():
    final = [np.mean(r['episode_rewards'][-100:]) for r in runs]
    print(f'  {name}: {np.mean(final):.1f} +/- {np.std(final):.1f}')

---

## Summary

Fill in after running all experiments.

### A.1 — MAPPO on simple_spread

*(Describe the learning curve. Does the team reward improve? How quickly? What does this tell you about parameter sharing?)*

---

### A.2 — MADDPG on simple_spread

*(Describe the learning curve. How does convergence speed compare to MAPPO? Did reward normalization and soft target updates affect stability?)*

---

### B.1 — MAPPO vs MADDPG

| Algorithm | Final Team Reward (mean ± std, 3 seeds) |
|-----------|------------------------------------------|
| MAPPO  | *(fill in after running B.1)* |
| MADDPG | *(fill in after running B.1)* |

*(Which algorithm performs better on this cooperative task? Why? Consider: parameter sharing, on-policy vs off-policy, stability.)*

---

### B.2 — MAPPO Entropy Coefficient Ablation

| ent_coef | Final Team Reward (mean ± std, 3 seeds) |
|----------|------------------------------------------|
| 0.0      | *(fill in)* |
| 0.001    | *(fill in)* |
| 0.01     | *(fill in)* |
| 0.05     | *(fill in)* |

*(Which value works best? What happens at the extremes? Why is a small entropy bonus generally better than zero or a large value?)*

---

**Lab designed by Amath Sow:** [amath.sow@liu.se](mailto:amath.sow@liu.se)

**TDDE78 — Deep Reinforcement Learning, Linköping University — Spring 2026**